# 00 · Validação: o método recupera a aversão a risco?

Antes de usar dados reais, testo o método num market maker simulado em que eu **sei** o valor da aversão a risco (γ). Se o estimador não acertar aqui, não faz sentido confiar nele na Hyperliquid.

**O modelo.** Uso a versão de horizonte longo do modelo de Avellaneda-Stoikov (Guéant, Lehalle e Fernandez-Tapia). Ele diz duas coisas sobre um market maker que tem medo de carregar estoque:

1. **Deslocamento (skew):** quanto mais inventário ele tem, mais ele baixa as duas cotações para vender logo. O centro das cotações fica em `mid − q·c2`, e `c2` cresce com γ.
2. **Spread:** a distância entre compra e venda é `2·c1 + Δ·c2`, onde `c1` depende quase só de quão rápido as ordens são executadas.

Então existem **duas formas de medir γ**: pelo deslocamento e pelo spread. Se o modelo estiver certo, as duas dão o mesmo número.

O que preciso estimar dos dados: a volatilidade do preço (σ), a intensidade de execução das ordens (A e k, na fórmula `λ(δ) = A·e^(−k·δ)`) e o tamanho do lote (Δ).

In [ ]:
import sys; sys.path.insert(0, "../src")
import warnings; warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from fear_gauge.glft import MarketParams
from fear_gauge.simulate import simulate_mm
from fear_gauge.estimate import estimate_gamma, fit_intensity, rolling_gauge, model_quotes_check
from fear_gauge import breaks, plots

pd.set_option("display.float_format", "{:.4g}".format)

## 1. Um market maker de livro-texto

γ verdadeiro = 0,5. Seis horas de simulação, um registro por segundo.

In [ ]:
P = MarketParams(sigma=0.02, A=1.0, k=100.0, delta=1.0)
sim = simulate_mm(0.5, P, seconds=6 * 3600, seed=11)
est = estimate_gamma(sim)
est.as_series()

In [ ]:
plots.skew_vs_inventory(sim, est)
plt.show()
plots.intensity_curve(sim, fit_intensity(sim))
plt.show()

## 2. As duas formas de medir γ são igualmente boas?

Rodo 10 simulações para cada valor de γ e comparo as duas estimativas.

In [ ]:
linhas = []
for gamma in (0.1, 0.5, 2.0):
    for seed in range(10):
        e = estimate_gamma(simulate_mm(gamma, P, seed=100 + seed))
        linhas.append({"gamma_real": gamma, "pelo_deslocamento": e.gamma_skew, "pelo_spread": e.gamma_spread})
mc = pd.DataFrame(linhas)
mc.groupby("gamma_real").agg(
    deslocamento_media=("pelo_deslocamento", "mean"), deslocamento_desvio=("pelo_deslocamento", "std"),
    spread_media=("pelo_spread", "mean"), spread_desvio=("pelo_spread", "std"),
    spread_sem_solucao=("pelo_spread", lambda x: x.isna().mean()))

## 3. O fear gauge percebe quando o medo muda?

Na primeira metade γ = 0,3 e na segunda γ = 2,0. Estimo γ em janelas de 1 hora andando de 15 em 15 minutos.

In [ ]:
g_real = np.r_[np.full(3 * 3600, 0.3), np.full(3 * 3600, 2.0)]
sim_regime = simulate_mm(g_real, P, seed=4)
gauge = rolling_gauge(sim_regime)
plots.gauge(gauge, true_gamma=pd.Series(g_real, index=sim_regime.index).resample("15min").last())
plt.show()

## 4. Os testes de quebra detectam cada problema?

Crio três versões do market maker que fogem do livro-texto e vejo se o teste certo acende em cada uma.

In [ ]:
cenarios = {
    "livro-texto": {},
    "seleção adversa (preço anda contra depois do fill)": {"adverse_jump": 0.01},
    "limite de inventário (para de cotar em |q| = 3)": {"inventory_cap": 3},
    "tick de 1 centavo": {"tick": 0.01},
}
linhas = []
for nome, kw in cenarios.items():
    df = simulate_mm(0.5, P, seed=11, **kw)
    e = estimate_gamma(df)
    m = breaks.markouts(df, horizons=(1,))
    faixas, _ = breaks.skew_shape(df)
    linhas.append({"cenário": nome, "γ pelo deslocamento": e.gamma_skew, "γ pelo spread": e.gamma_spread,
                   "markout 1s": m["movimento_medio"].mean(), "p-valor markout": m["pvalor"].max(),
                   "tempo cotando só 1 lado": (faixas["so_bid"] + faixas["so_ask"]).mul(faixas["segundos"]).sum() / faixas["segundos"].sum()})
pd.DataFrame(linhas).set_index("cenário")

## 5. Conclusões

**Resumo:** o método funciona, mas só por um dos caminhos. O γ medido pelo deslocamento das cotações é confiável. O γ medido pelo spread, não.

### O que funcionou
- **γ pelo deslocamento acerta.** Nas 30 simulações, a média ficou entre 2% e 4% do valor verdadeiro, com erro típico de 4% a 10%.
- **O fear gauge acompanha a mudança.** Quando o γ verdadeiro sobe de 0,3 para 2,0, a estimativa em janela móvel sobe junto, com atraso de mais ou menos uma janela.
- **Cada teste detecta o seu problema.** O markout fica perto de zero no livro-texto e cai para cerca de −0,007 com p-valor zero quando existe seleção adversa. O limite de inventário aparece como cerca de 8% do tempo cotando só de um lado.

### O que não funcionou
- **γ pelo spread é instável.** O desvio fica entre 0,5 e 0,7, maior que o próprio γ em vários casos, e com γ = 0,1 a equação não tem solução em 10% das simulações. O motivo é que o spread depende quase todo de `1/k`, ou seja, da velocidade de execução, e quase nada da aversão a risco.
- **Com tick mínimo, o γ pelo spread explode.** Com tick de 1 centavo, ele foi para cerca de 25, quando o verdadeiro é 0,5, porque o arredondamento alarga o spread. O γ pelo deslocamento continuou perto de 0,45.

### O que isso significa para os dados reais
Vou usar o **deslocamento** como medida principal de γ. A comparação com o spread fica como teste do modelo: se as duas medidas discordarem muito, é sinal de que o modelo de livro-texto não descreve esse market maker.